In [1]:
%load_ext autoreload
%autoreload 2

# --- Standard libs ---
import numpy as np
import pandas as pd
import time
import sys
import serial
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# --- Project imports ---
from instruments.gilson.gilson_ethernet import GilsonEthernet
from instruments.gilson.tray import Tray
from instruments.gilson.rack import Rack_209, Rack_3dp
from instruments.gilson.probe import ProbeState    # NEW
from core.logging import flow_logger as logger
from core.tracing import *
from instruments.vici.dim import DIM
from instruments.vici.fsm import FSM
from instruments.syr_pumps.HB_syr_pump import HBElite
from instruments.knauer.knauer_pump_azura import KnauerPumpAzura
from ecosystems.reaction_segment_generation import RSG
from ecosystems.segmentation import Segmentation


# Experiment framework
from core.experiment_context import ExperimentManager
from core.experiment_builder import ExperimentBuilder
from core.experiment_validator import ExperimentValidator
from core.experiment_compiler import ExperimentCompiler
from core.experiment_intent import ExperimentIntent
from core.inventory import Inventory

In [2]:
# --- Syringe pump ---
syr_pump = HBElite("COM10")
syr_pump.connect()

# --- DIM ---
dim = DIM("COM5")
dim.connect()

# --- FSM ---
fsm = FSM("COM2")
fsm.connect()

# --- Carrier pump ---
k_pump = KnauerPumpAzura("COM4")
k_pump.connect()

HB Elite pump connected on COM10
Connected to DIM on COM5
Connected to FSM on COM2
Connected to Azura pump on COM4


In [3]:
# --- Instantiate the tray ---
tray = Tray()

# --- Instantiate modules for the tray ---
rack1 = Rack_209()
rack2 = Rack_3dp()

# --- Assign modules to tray slots ---
tray.assign_slot(1, rack1, alias = "rack1")    # Assigns rack1 to slot 1
tray.assign_slot(2, rack2, alias = "rack2")    # Assigns rack2 to slot 2
tray.assign_slot(3, dim, alias = "dim")      # Assigns dim to slot 3

In [4]:
# --- Connect to Gilson ---
g = GilsonEthernet("192.168.1.101", admin_port=50185, tray=tray)

# --- Tell gilson instance about the DIM ---
g.dim = dim

# --- Instantiate the probe (state machine only) ---
probe = ProbeState()

# --- Start logging (declare this run belongs to the experiment "xxxxx") ---
logger.start_experiment("VB-1E-9")

In [5]:
# Instantiate the RSG ecosystem with the connected Gilson, pump, DIM and probe
rsg = RSG(gilson=g, pump=syr_pump, dim=dim, probe=probe)

# Instantiate the segmentation ecosystem with the dim, runze valve, knauer pump and RSG connected
seg = Segmentation(dim=dim, fsm=fsm, carrier_pump=k_pump, rsg=rsg)

[Pump] Flow stopped
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT


In [6]:
inventory = Inventory()

# Clear + reassign (optional but safer for now)
inventory.assign(
    module="rack2",
    vial=1,
    name="MeCN",
    concentration_M=None,
    solvent="MeCN",
    current_volume_uL=40000,
    min_safe_volume_uL=20000
)

inventory.assign(
    module="rack1",
    vial=1,
    name="BnOH",
    concentration_M=0.0428,
    solvent="MeCN",
    current_volume_uL=1500,
    min_safe_volume_uL=500
)

inventory.assign(
    module="rack1",
    vial=2,
    name="BnOH",
    concentration_M=0.0428,
    solvent="MeCN",
    current_volume_uL=1500,
    min_safe_volume_uL=500,
)

In [7]:
intent = ExperimentIntent(
    experiment_id="VB-1E-9_test",
    description="First 2 slug test ahead of VB-1E-9"
)

intent.add_block(
    name="Two BnOH slugs",
    components= [
        {
            "name":"BnOH",
            "concentration_M": 0.0428,
            "solvent": "MeCN",
        }
    ],
    ratios=[[100] for _ in range (2)],  
    total_volume_uL=100.0
)

intent.summary()

{'experiment_id': 'VB-1E-9_test',
 'description': 'First 2 slug test ahead of VB-1E-9',
 'num_blocks': 1,
 'estimated_slugs': 2,
 'blocks': [{'name': 'Two BnOH slugs',
   'components': [{'name': 'BnOH',
     'concentration_M': 0.0428,
     'solvent': 'MeCN'}],
   'num_slugs': 2,
   'total_volume_uL': 100.0,
   'fixed_total_volume': True}]}

In [8]:
compiler = ExperimentCompiler(inventory, trace=True)

compiled_df = compiler.compile(intent)

compiled_df


[RESOLVE] BnOH | request=100.0 µL
  candidate rack1:1: usable=1000.0, remaining=1000.0
  candidate rack1:2: usable=1000.0, remaining=1000.0
  ✅ selected rack1:1 (vial 1) | remaining before=1000.0
  → remaining after=900.0

[RESOLVE] BnOH | request=100.0 µL
  candidate rack1:1: usable=1000.0, remaining=900.0
  candidate rack1:2: usable=1000.0, remaining=1000.0
  ✅ selected rack1:1 (vial 1) | remaining before=900.0
  → remaining after=800.0
[COMPILER OUTPUT SHAPE] (2, 10)
[COMPILER OUTPUT COLUMNS] ['slug_id', 'module', 'vial', 'volume_uL', 'component', 'concentration_M', 'solvent', 'block_id', 'slug_order', 'component_order']
[COMPILER SAMPLE ROW] {'slug_id': 'block_1_slug_1', 'module': 'rack1', 'vial': 1, 'volume_uL': 100.0, 'component': 'BnOH', 'concentration_M': 0.0428, 'solvent': 'MeCN', 'block_id': 'block_1', 'slug_order': 1, 'component_order': 1}


,slug_id,module,vial,volume_uL,component,concentration_M,solvent,block_id,slug_order,component_order
0,block_1_slug_1,rack1,1,100.0,BnOH,0.0428,MeCN,block_1,1,1
1,block_1_slug_2,rack1,1,100.0,BnOH,0.0428,MeCN,block_1,2,1


In [9]:
builder = ExperimentBuilder(inventory=inventory)

result = builder.build_and_create(
    experiment_id="VB-1E-9_test",
    rows=compiled_df,
    description=intent.description,
    global_conditions={
        "flowrate_ul_min": 1000,
        "gas_prime_s": 20,
        "withdraw_rate_ml_min": 0.25,
        "dispense_rate_ml_min": 0.5,
        "needle_wash_volume_ul": 80.0,
        "between_slug_wash_volume_ul": 100.0,
    },
    overwrite=True
)

print(result["plan_path"])
plan = result["plan"]

pd.DataFrame(result["summary"])

C:\Users\CHAD-HPLC\Documents\VictorFlow\Experiments\VB-1E-9_test\plan.json


,slug_id,num_components,total_volume_uL,components
0,block_1_slug_1,1,100.0,"[(BnOH, rack1, 1, 100.0)]"
1,block_1_slug_2,1,100.0,"[(BnOH, rack1, 1, 100.0)]"


In [10]:
manager = ExperimentManager()

manager.load_experiment("VB-1E-9_test")

manager.status()

[EXPERIMENT] VB-1E-9_test loaded
[EXPERIMENT] Plan ready
[EXPERIMENT] Log ready
[EXPERIMENT] State = prepared
[EXPERIMENT] Awaiting arm_experiment()
Mode: experiment
Experiment: VB-1E-9_test
State: prepared
System state: UNKNOWN
Reactor state: UNKNOWN
RSG state: UNKNOWN
Platform state: UNKNOWN
Slug index: 0


In [30]:
manager.precondition_reactor(seg, carrier_flowrate_ul_min=1000, stabilisation_time_s=60)

manager.prepare_rsg(
    seg,
    air_gap=20.0,
    rate_wdr=0.10
)

[REACTOR] Setting valve positions
[FSM] Already at B -> state = CARRIER_TO_DIM
[DIM] Valve already at A -> state = DIMState.INJECT
[k_pump] Flow rate set to 1000 uL/min
[Pump] Flow started
[REACTOR] Stabilising...
[SYSTEM] Reactor READY
[RSG] Initialising
[Initialise] Setting up start condition
[Air Gap] 20.0uL @ 0.1mL/min
[Probe] [working_fluid] [air (20.0 uL)]
[Initialise] Ready - air gap in place
[SYSTEM] RSG READY


In [11]:
manager.preview_execution()


========== EXPERIMENT PREVIEW ==========

Experiment: VB-1E-9_test
Description: First 2 slug test ahead of VB-1E-9

[GLOBAL CONDITIONS]
  Flowrate: 1000.0 uL/min
  Withdraw rate: 0.25 mL/min
  Dispense rate: 0.5 mL/min
  Gas prime: 20.0 s
  Needle wash: 80.0 uL
  Between slug wash: 100.0 uL

----------------------------------------

[SLUG 1] block_1_slug_1
  Load 100.0 uL of BnOH from rack1 vial 1
  Reaction volume: 100.0 uL
  Estimated injected slug: 110.0 uL

[SLUG 2] block_1_slug_2
  Load 100.0 uL of BnOH from rack1 vial 1
  Reaction volume: 100.0 uL
  Estimated injected slug: 110.0 uL




In [32]:
manager.arm_experiment()

Exception: Cannot arm experiment while state = completed

In [ ]:
manager.execute_experiment(
    seg,
    wash_policy="needle_only",
)